# Supply Chain Shipment Price Analysis

**Goal:** Explore supply-chain shipment, pricing, logistics, and delivery patterns using Python.

**Dataset source:** Kaggle — *Supply Chain Shipment Pricing Data*  
Expected file: `data/SCMS_Delivery_History_Dataset.csv`

This notebook is an original portfolio workflow built around the public dataset structure.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
DATA_PATH = Path("data/SCMS_Delivery_History_Dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Download SCMS_Delivery_History_Dataset.csv from the Kaggle source "
        "listed in README.md and place it inside the data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 1. Data Understanding


In [ ]:
print("Columns:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:>2}. {c}")

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Missing-value profile
missing = (
    df.isna().sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_pct", ascending=False)
)
missing.head(15)


## 2. Basic Cleaning


In [ ]:
clean = df.copy()

# Trim text columns
for c in clean.select_dtypes(include="object").columns:
    clean[c] = clean[c].astype(str).str.strip().replace({"nan": np.nan})

# Convert likely date columns where possible
date_cols = [c for c in clean.columns if "date" in c.lower()]
for c in date_cols:
    clean[c] = pd.to_datetime(clean[c], errors="coerce")

print("Detected date columns:", date_cols)
print("Rows after cleaning:", len(clean))


## 3. Shipment Overview


In [ ]:
def top_counts(column, n=10):
    if column not in clean.columns:
        print(f"{column!r} not found in this dataset.")
        return pd.DataFrame()
    return clean[column].value_counts(dropna=False).head(n).to_frame("count")

display(top_counts("Country", 10))
display(top_counts("Shipment Mode", 10))
display(top_counts("Fulfill Via", 10))


In [ ]:
# Top countries by shipment records
if "Country" in clean.columns:
    counts = clean["Country"].value_counts().head(10).sort_values()
    plt.figure(figsize=(9,5))
    plt.barh(counts.index, counts.values)
    plt.title("Top 10 Countries by Shipment Records")
    plt.xlabel("Shipment records")
    plt.tight_layout()
    plt.show()

# Shipment mode distribution
if "Shipment Mode" in clean.columns:
    counts = clean["Shipment Mode"].value_counts().sort_values()
    plt.figure(figsize=(8,4))
    plt.barh(counts.index.astype(str), counts.values)
    plt.title("Shipment Mode Distribution")
    plt.xlabel("Shipment records")
    plt.tight_layout()
    plt.show()


## 4. Pricing and Cost Analysis


In [ ]:
# Discover numeric business columns that may represent price, value, freight, cost, quantity, or weight
business_terms = ("price", "value", "freight", "cost", "quantity", "weight")
business_cols = [
    c for c in clean.columns
    if any(term in c.lower() for term in business_terms)
]

print("Potential business-value columns:")
for c in business_cols:
    print("-", c)

numeric_business = clean[business_cols].select_dtypes(include=np.number) if business_cols else pd.DataFrame()
numeric_business.describe().T if not numeric_business.empty else "No numeric business columns detected."


In [ ]:
# Common columns in this dataset; code runs only when they exist.
for col in ["Unit Price", "Line Item Quantity", "Line Item Value", "Freight Cost (USD)", "Weight (Kilograms)"]:
    if col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

summary_cols = [c for c in ["Unit Price", "Line Item Quantity", "Line Item Value",
                            "Freight Cost (USD)", "Weight (Kilograms)"] if c in clean.columns]
if summary_cols:
    display(clean[summary_cols].describe().T)


## 5. Country and Product Value Analysis


In [ ]:
if {"Country", "Line Item Value"}.issubset(clean.columns):
    country_value = (
        clean.groupby("Country", dropna=False)["Line Item Value"]
             .sum()
             .sort_values(ascending=False)
             .head(10)
    )
    display(country_value.to_frame("total_line_item_value"))

    plt.figure(figsize=(9,5))
    plt.barh(country_value.sort_values().index, country_value.sort_values().values)
    plt.title("Top 10 Countries by Line Item Value")
    plt.xlabel("Total Line Item Value")
    plt.tight_layout()
    plt.show()

for col in ["Vendor", "Brand", "Product Group", "Sub Classification"]:
    if col in clean.columns:
        print(f"\nTop {col}:")
        display(clean[col].value_counts().head(10).to_frame("shipment_records"))


## 6. Delivery Performance


In [ ]:
# Create delivery delay when both date fields exist.
scheduled_candidates = [c for c in clean.columns if "scheduled" in c.lower() and "date" in c.lower()]
delivered_candidates = [c for c in clean.columns if "delivered" in c.lower() and "date" in c.lower()]

print("Scheduled-date candidates:", scheduled_candidates)
print("Delivered-date candidates:", delivered_candidates)

if scheduled_candidates and delivered_candidates:
    scheduled_col = scheduled_candidates[0]
    delivered_col = delivered_candidates[0]
    clean["Delivery_Delay_Days"] = (clean[delivered_col] - clean[scheduled_col]).dt.days

    display(clean["Delivery_Delay_Days"].describe())

    plt.figure(figsize=(8,4))
    clean["Delivery_Delay_Days"].dropna().clip(-100, 200).hist(bins=30)
    plt.title("Delivery Delay Distribution")
    plt.xlabel("Days (positive = late)")
    plt.ylabel("Shipments")
    plt.tight_layout()
    plt.show()


## 7. Correlation Check


In [ ]:
num = clean.select_dtypes(include=np.number)
if num.shape[1] >= 2:
    corr = num.corr(numeric_only=True)
    display(corr.round(2))
else:
    print("Not enough numeric columns for correlation analysis.")


## 8. Portfolio Conclusions

Use the executed results to write 4–6 evidence-based findings, for example:

- Which countries account for the most shipment activity?
- Which shipment mode is used most frequently?
- Which countries or products generate the highest line-item value?
- Are there noticeable delivery delays?
- Which vendors / brands dominate the shipment history?
- What business actions could reduce cost or delivery risk?

> Replace these prompts with actual findings after running the notebook.
